In [2]:
# Optional: install dependencies (uncomment if needed)
# !pip install torch torchvision scikit-learn tqdm
import os
os.environ.setdefault('MKL_THREADING_LAYER', 'GNU')
import time
import math
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from tqdm import tqdm

In [3]:
# Reproducible settings and synthetic data generator
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

def spectral_field(num_samples, dim, alpha=1.5, freq_scale=0.05, noise_std=1.0, rng=None):
    """Generate smooth correlated static signals by filtering white noise in Fourier space."""
    rng = rng or np.random.default_rng()
    freqs = np.fft.rfftfreq(dim)
    safe_freqs = np.maximum(freqs, 1e-4)
    filt = 1.0 / (1.0 + (safe_freqs / freq_scale) ** alpha)
    coeff_real = rng.normal(scale=noise_std, size=(num_samples, freqs.size))
    coeff_imag = rng.normal(scale=noise_std, size=(num_samples, freqs.size))
    coeffs = (coeff_real + 1j * coeff_imag) * filt
    coeffs[:, 0].imag = 0.0  # enforce real signal
    fields = np.fft.irfft(coeffs, n=dim, axis=1)
    fields = fields.astype(np.float32)
    fields -= fields.mean(axis=1, keepdims=True)
    fields /= fields.std(axis=1, keepdims=True) + 1e-6
    return fields

def stochastic_oscillator(num_samples, dim, harmonics=3, freq_range=(0.5, 3.5), noise_std=0.4, rng=None):
    """Simulate bursty temporal telemetry via random harmonic oscillators with injected noise."""
    rng = rng or np.random.default_rng()
    time = np.linspace(0.0, 1.0, dim, dtype=np.float32)
    signals = np.zeros((num_samples, dim), dtype=np.float32)
    for h in range(1, harmonics + 1):
        freqs = rng.uniform(freq_range[0], freq_range[1], size=(num_samples, 1))
        phases = rng.uniform(0.0, 2 * np.pi, size=(num_samples, 1))
        amps = rng.normal(scale=1.0 / h, size=(num_samples, 1))
        oscillation = np.sin(2 * np.pi * (freqs * time + phases))
        signals += (amps * oscillation).astype(np.float32)
    noise = rng.normal(scale=noise_std, size=signals.shape).astype(np.float32)
    signals += noise
    signals -= signals.mean(axis=1, keepdims=True)
    signals /= signals.std(axis=1, keepdims=True) + 1e-6
    return signals

def network_feature_block(num_samples, dim, base_rate=1.2, tail_strength=0.15, rng=None):
    """Create heavy-tailed network traffic descriptors with occasional beaconing bursts."""
    rng = rng or np.random.default_rng()
    log_means = rng.normal(loc=base_rate, scale=0.4, size=(num_samples, dim))
    log_stds = rng.uniform(0.2, 0.6, size=(num_samples, dim))
    counts = rng.lognormal(mean=log_means, sigma=log_stds).astype(np.float32)
    bursts = rng.binomial(1, tail_strength, size=(num_samples, dim))
    burst_magnitudes = rng.lognormal(mean=3.0, sigma=0.5, size=(num_samples, dim)).astype(np.float32)
    counts += bursts * burst_magnitudes
    counts = np.log1p(counts)
    counts -= counts.mean(axis=1, keepdims=True)
    counts /= counts.std(axis=1, keepdims=True) + 1e-6
    return counts.astype(np.float32)

def cfg_api_graph_features(num_samples, dim, base_dim=256, cfg_nodes=(25, 90), api_bins=64, embed_dim=64, steps=3, mix_matrix=None, rng=None):
    """Approximate CFG/API graph embeddings via message passing and histogram summaries."""
    rng = rng or np.random.default_rng()
    deg_bins = np.linspace(0, 12, 17)
    depth_bins = np.linspace(0, 16, 17)
    base_feats = np.zeros((num_samples, base_dim), dtype=np.float32)
    for idx in range(num_samples):
        n_cfg = int(rng.integers(cfg_nodes[0], cfg_nodes[1] + 1))
        edge_prob = rng.uniform(0.06, 0.16)
        upper = rng.random((n_cfg, n_cfg))
        adj = np.triu((upper < edge_prob).astype(np.float32), k=1)
        # enforce entry/exit corridors so the graph resembles a CFG rather than Erdos-Renyi noise
        if n_cfg > 2:
            entry = rng.integers(0, max(1, n_cfg // 6))
            exit_start = max(entry + 1, n_cfg // 2)
            exit_node = rng.integers(exit_start, n_cfg)
            adj[entry, entry + 1 : min(entry + 4, n_cfg)] = 1.0
            adj[max(0, exit_node - 3) : exit_node, exit_node - 1] = 1.0
        out_deg = adj.sum(axis=1)
        in_deg = adj.sum(axis=0)
        deg_hist_out, _ = np.histogram(np.clip(out_deg, 0, deg_bins[-1] - 1e-3), bins=deg_bins, density=True)
        deg_hist_in, _ = np.histogram(np.clip(in_deg, 0, deg_bins[-1] - 1e-3), bins=deg_bins, density=True)
        depths = np.zeros(n_cfg, dtype=np.float32)
        for node in range(n_cfg):
            preds = np.nonzero(adj[:node, node])[0]
            depths[node] = 0.0 if preds.size == 0 else depths[preds].max() + 1.0
        depth_hist, _ = np.histogram(np.clip(depths, 0, depth_bins[-1] - 1e-3), bins=depth_bins, density=True)
        node_state = rng.normal(scale=1.0, size=(n_cfg, embed_dim)).astype(np.float32)
        trans = adj.T
        for _ in range(steps):
            inflow = trans @ node_state
            denom = in_deg[:, None] + 1.0
            node_state = np.tanh(node_state + inflow / denom)
        graph_mean = node_state.mean(axis=0)
        graph_var = node_state.var(axis=0)
        n_api = int(rng.integers(20, 90))
        api_calls = rng.integers(0, api_bins, size=n_api)
        api_hist = np.bincount(api_calls, minlength=api_bins).astype(np.float32)
        api_hist /= api_hist.sum() + 1e-6
        feature_vec = np.concatenate([
            graph_mean,
            graph_var,
            deg_hist_out,
            deg_hist_in,
            depth_hist,
            api_hist,
        ])
        if feature_vec.size < base_dim:
            feature_vec = np.pad(feature_vec, (0, base_dim - feature_vec.size))
        else:
            feature_vec = feature_vec[:base_dim]
        feature_vec -= feature_vec.mean()
        feature_vec /= feature_vec.std() + 1e-6
        base_feats[idx] = feature_vec.astype(np.float32)
    if mix_matrix is not None:
        mixed = base_feats @ mix_matrix
        mixed -= mixed.mean(axis=1, keepdims=True)
        mixed /= mixed.std(axis=1, keepdims=True) + 1e-6
        return mixed.astype(np.float32)
    base_feats -= base_feats.mean(axis=1, keepdims=True)
    base_feats /= base_feats.std(axis=1, keepdims=True) + 1e-6
    return base_feats.astype(np.float32)

# Feature dimensions (adjust to match your real data)
static_dim = 256        # static features vector length
dynamic_dim = 128       # dynamic/text-encoded feature vector length
graphical_dim = 512     # graphical features vector length (pre-extracted)
network_dim = 200       # network traffic feature vector length
GRAPH_BASE_DIM = 256
graph_mix = rng.normal(scale=1.0 / np.sqrt(GRAPH_BASE_DIM), size=(GRAPH_BASE_DIM, graphical_dim)).astype(np.float32)

# Dataset size and classes
N_SAMPLES = 2000
NUM_CLASSES = 2      # binary classification (malware / benign)

# Generate synthetic features with modality-specific stochastic processes
X_static = spectral_field(
    num_samples=N_SAMPLES,
    dim=static_dim,
    alpha=1.8,
    freq_scale=0.04,
    noise_std=0.7,
    rng=rng
).astype(np.float32)

X_dynamic = stochastic_oscillator(
    num_samples=N_SAMPLES,
    dim=dynamic_dim,
    harmonics=4,
    freq_range=(0.8, 3.5),
    noise_std=0.45,
    rng=rng
)

X_graphical = cfg_api_graph_features(
    num_samples=N_SAMPLES,
    dim=graphical_dim,
    base_dim=GRAPH_BASE_DIM,
    cfg_nodes=(30, 90),
    api_bins=64,
    embed_dim=64,
    steps=3,
    mix_matrix=graph_mix,
    rng=rng
)

X_network = network_feature_block(
    num_samples=N_SAMPLES,
    dim=network_dim,
    base_rate=1.1,
    tail_strength=0.2,
    rng=rng
)

# Latent risk weights tie modalities to labels so the classification task is meaningful
static_w = rng.normal(scale=0.12, size=static_dim).astype(np.float32)
dynamic_w = rng.normal(scale=0.12, size=dynamic_dim).astype(np.float32)
graph_w = rng.normal(scale=0.12, size=graphical_dim).astype(np.float32)
network_w = rng.normal(scale=0.12, size=network_dim).astype(np.float32)
bias_term = -0.2

latent_score = (
    (X_static @ static_w) / np.sqrt(static_dim)
    + (X_dynamic @ dynamic_w) / np.sqrt(dynamic_dim)
    + (X_graphical @ graph_w) / np.sqrt(graphical_dim)
    + (X_network @ network_w) / np.sqrt(network_dim)
)
latent_score += 0.35 * X_network[:, : network_dim // 5].mean(axis=1)  # emphasize beacon-like behavior
latent_score += rng.normal(scale=0.6, size=N_SAMPLES)
logits = latent_score + bias_term
probs = 1.0 / (1.0 + np.exp(-logits))
y = (rng.random(N_SAMPLES) < probs).astype(np.int64)

print('Class balance -> malicious ratio:', y.mean().round(3))

# Quick train/val/test split
ids = np.arange(N_SAMPLES)
train_ids, test_ids = train_test_split(ids, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
train_ids, val_ids = train_test_split(train_ids, test_size=0.2, random_state=RANDOM_SEED, stratify=y[train_ids])

def slice_by_ids(ids_array):
    return dict(
        static=X_static[ids_array],
        dynamic=X_dynamic[ids_array],
        graphical=X_graphical[ids_array],
        network=X_network[ids_array],
        labels=y[ids_array]
)

train_data = slice_by_ids(train_ids)
val_data = slice_by_ids(val_ids)
test_data = slice_by_ids(test_ids)

print('Train/Val/Test sizes:', len(train_ids), len(val_ids), len(test_ids))

Class balance -> malicious ratio: 0.436
Train/Val/Test sizes: 1280 320 400


In [4]:
# Dataset and DataLoader wrappers
class MalwareDataset(Dataset):
    """Simple mapping from numpy feature blocks into torch tensors per modality."""
    def __init__(self, data_dict):
        self.static = torch.from_numpy(data_dict['static'])
        self.dynamic = torch.from_numpy(data_dict['dynamic'])
        self.graphical = torch.from_numpy(data_dict['graphical'])
        self.network = torch.from_numpy(data_dict['network'])
        self.labels = torch.from_numpy(data_dict['labels']).long()
    def __len__(self):
        return self.labels.shape[0]
    def __getitem__(self, idx):
        return dict(
            static=self.static[idx],
            dynamic=self.dynamic[idx],
            graphical=self.graphical[idx],
            network=self.network[idx],
            label=self.labels[idx]
)

batch_size = 64
train_ds = MalwareDataset(train_data)
val_ds = MalwareDataset(val_data)
test_ds = MalwareDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

print('Sample batch shapes:')
batch = next(iter(train_loader))
print('static', batch['static'].shape)
print('dynamic', batch['dynamic'].shape)
print('graphical', batch['graphical'].shape)
print('network', batch['network'].shape)

Sample batch shapes:
static torch.Size([64, 256])
dynamic torch.Size([64, 128])
graphical torch.Size([64, 512])
network torch.Size([64, 200])


In [5]:
# Model helpers and ViT-style multimodal transformer
class PatchEmbed(nn.Module):
    """Project dense feature vectors into patch tokens similar to ViT patch embeddings."""
    def __init__(self, patch_dim, embed_dim):
        super().__init__()
        self.patch_dim = patch_dim
        self.proj = nn.Linear(patch_dim, embed_dim)
    def forward(self, x):
        b, d = x.shape
        pad = (-d) % self.patch_dim
        if pad:
            x = F.pad(x, (0, pad))
        patches = x.view(b, -1, self.patch_dim)
        return self.proj(patches)

class VisionInspiredFusionTransformer(nn.Module):
    """Deeper transformer fusion block with reasoning tokens and modality gating."""
    def __init__(self, static_dim, dynamic_dim, graphical_dim, network_dim,
                 embed_dim=192, depth=6, n_heads=8, dropout=0.1, num_classes=2):
        super().__init__()
        self.embed_dim = embed_dim
        self.dropout = nn.Dropout(dropout)
        self.modality_order = ['static', 'dynamic', 'graphical', 'network']
        patch_config = {
            'static': {'dim': static_dim, 'patch': 32},
            'dynamic': {'dim': dynamic_dim, 'patch': 16},
            'graphical': {'dim': graphical_dim, 'patch': 32},
            'network': {'dim': network_dim, 'patch': 20},
        }
        self.patchers = nn.ModuleDict({name: PatchEmbed(cfg['patch'], embed_dim) for name, cfg in patch_config.items()})
        self.token_counts = {}
        total_tokens = 2  # CLS + REASON tokens
        for name, cfg in patch_config.items():
            tokens = math.ceil(cfg['dim'] / cfg['patch'])
            self.token_counts[name] = tokens
            total_tokens += tokens
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.reason_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, total_tokens, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.modality_gate = nn.Parameter(torch.zeros(len(self.modality_order)))
        self.reason_context_vectors = nn.Parameter(torch.randn(len(self.modality_order), embed_dim))
        self.classifier = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, num_classes))
        self.reason_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, len(self.modality_order)))

    def _embed_modalities(self, static, dynamic, graphical, network):
        tensors = {'static': static, 'dynamic': dynamic, 'graphical': graphical, 'network': network}
        token_blocks = []
        for idx, name in enumerate(self.modality_order):
            tokens = self.patchers[name](tensors[name])  # (B, tokens, E)
            gate = torch.sigmoid(self.modality_gate[idx])
            token_blocks.append(self.dropout(tokens * gate))
        return token_blocks

    def _forward_with_reason(self, static, dynamic, graphical, network):
        batch_size = static.size(0)
        token_blocks = self._embed_modalities(static, dynamic, graphical, network)
        cls = self.cls_token.expand(batch_size, -1, -1)
        reason = self.reason_token.expand(batch_size, -1, -1)
        seq = torch.cat([cls, reason] + token_blocks, dim=1)
        seq = seq + self.pos_embed[:, :seq.shape[1], :]
        fused = self.transformer(seq)
        cls_out = fused[:, 0, :]
        reason_out = fused[:, 1, :]
        logits = self.classifier(cls_out)

        # Aggregate modality evidence for reasoning scores
        offset = 2
        pooled = []
        for idx, name in enumerate(self.modality_order):
            tokens = fused[:, offset: offset + self.token_counts[name], :]
            pooled.append(tokens.mean(dim=1))
            offset += self.token_counts[name]
        pooled = torch.stack(pooled, dim=1)  # (B, M, E)
        context_scores = torch.einsum('bme,me->bm', pooled, self.reason_context_vectors)
        reason_logits = self.reason_head(reason_out) + context_scores
        return logits, reason_logits

    def forward(self, static, dynamic, graphical, network):
        logits, _ = self._forward_with_reason(static, dynamic, graphical, network)
        return logits

    def explain(self, static, dynamic, graphical, network):
        self.eval()
        with torch.no_grad():
            _, reason_logits = self._forward_with_reason(static, dynamic, graphical, network)
            return torch.softmax(reason_logits, dim=-1)

# Instantiate model and move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = VisionInspiredFusionTransformer(
    static_dim, dynamic_dim, graphical_dim, network_dim, num_classes=NUM_CLASSES
).to(device)
print(model)

VisionInspiredFusionTransformer(
  (dropout): Dropout(p=0.1, inplace=False)
  (patchers): ModuleDict(
    (static): PatchEmbed(
      (proj): Linear(in_features=32, out_features=192, bias=True)
    )
    (dynamic): PatchEmbed(
      (proj): Linear(in_features=16, out_features=192, bias=True)
    )
    (graphical): PatchEmbed(
      (proj): Linear(in_features=32, out_features=192, bias=True)
    )
    (network): PatchEmbed(
      (proj): Linear(in_features=20, out_features=192, bias=True)
    )
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=192, out_features=192, bias=True)
        )
        (linear1): Linear(in_features=192, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=768, out_features=192, bias=True)
        (norm1): LayerNorm((192,), eps=1e-05, el

In [6]:
# Training utilities
class_counts = np.bincount(y, minlength=NUM_CLASSES)
class_weights = torch.tensor(
    len(y) / (NUM_CLASSES * class_counts),
    dtype=torch.float32,
    device=device,
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)
amp_enabled = device.type == 'cuda'
scaler = torch.amp.GradScaler(device='cuda') if amp_enabled else None

def run_epoch(loader, train=True):
    model.train(train)
    if not train:
        model.eval()
    losses = []
    preds = []
    trues = []
    phase = 'train' if train else 'eval'
    pbar = tqdm(loader, desc=phase)
    for batch in pbar:
        static = batch['static'].to(device)
        dynamic = batch['dynamic'].to(device)
        graphical = batch['graphical'].to(device)
        network = batch['network'].to(device)
        labels = batch['label'].to(device)

        context = torch.autocast(device_type='cuda', dtype=torch.float16) if amp_enabled else nullcontext()
        with context:
            logits = model(static, dynamic, graphical, network)
            loss = criterion(logits, labels)
        if train:
            optimizer.zero_grad(set_to_none=True)
            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        losses.append(loss.item())
        batch_preds = logits.detach().cpu().numpy().argmax(axis=1)
        preds.extend(batch_preds.tolist())
        trues.extend(labels.detach().cpu().numpy().tolist())

    acc = accuracy_score(trues, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(trues, preds, average='binary', zero_division=0)
    return np.mean(losses), acc, precision, recall, f1

# Training loop
n_epochs = 12
best_val_f1 = 0.0
for epoch in range(1, n_epochs+1):
    start = time.time()
    train_loss, train_acc, train_prec, train_rec, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_prec, val_rec, val_f1 = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    elapsed = time.time() - start
    print(f'Epoch {epoch:02d} | Train loss {train_loss:.4f} acc {train_acc:.3f} f1 {train_f1:.3f} | ' +
          f'Val loss {val_loss:.4f} acc {val_acc:.3f} f1 {val_f1:.3f} | {elapsed:.1f}s')
    # checkpoint
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_multimodal_model.pt')
        print('Saved best model')

eval: 100%|██████████| 5/5 [00:00<00:00, 189.06it/s]


Epoch 01 | Train loss 0.8880 acc 0.495 f1 0.472 | Val loss 0.7211 acc 0.562 f1 0.000 | 0.7s


eval: 100%|██████████| 5/5 [00:00<00:00, 234.53it/s]



Epoch 02 | Train loss 0.7269 acc 0.495 f1 0.451 | Val loss 0.7163 acc 0.438 f1 0.609 | 0.3s
Saved best model


eval: 100%|██████████| 5/5 [00:00<00:00, 234.54it/s]


...

Epoch 10 | Train loss 0.6950 acc 0.459 f1 0.553 | Val loss 0.6946 acc 0.562 f1 0.000 | 0.3s


eval: 100%|██████████| 5/5 [00:00<00:00, 236.00it/s]


Epoch 11 | Train loss 0.6983 acc 0.505 f1 0.409 | Val loss 0.6940 acc 0.438 f1 0.609 | 0.3s


eval: 100%|██████████| 5/5 [00:00<00:00, 234.70it/s]

Epoch 12 | Train loss 0.6959 acc 0.483 f1 0.491 | Val loss 0.6942 acc 0.562 f1 0.000 | 0.3s


In [7]:
# Evaluation on test set with probabilities / ROC AUC
model.load_state_dict(torch.load('best_multimodal_model.pt', map_location=device))
model.eval()
all_preds = []
all_probs = []
all_trues = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='test'):
        static = batch['static'].to(device)
        dynamic = batch['dynamic'].to(device)
        graphical = batch['graphical'].to(device)
        network = batch['network'].to(device)
        labels = batch['label'].to(device)
        logits = model(static, dynamic, graphical, network)
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_trues.extend(labels.cpu().numpy().tolist())

acc = accuracy_score(all_trues, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_trues, all_preds, average='binary', zero_division=0)
try:
    auc = roc_auc_score(all_trues, all_probs)
except Exception:
    auc = float('nan')
print(f'Test Acc {acc:.4f} Prec {prec:.4f} Rec {rec:.4f} F1 {f1:.4f} AUC {auc:.4f}')

# Reasoning weights derived from the REASON token
sample_batch = next(iter(test_loader))
reason_scores = model.explain(
    sample_batch['static'][:1].to(device),
    sample_batch['dynamic'][:1].to(device),
    sample_batch['graphical'][:1].to(device),
    sample_batch['network'][:1].to(device),
)
reason_scores = reason_scores.squeeze().cpu().numpy()
modalities = ['static', 'dynamic', 'graphical', 'network']
reason_map = {name: float(reason_scores[idx]) for idx, name in enumerate(modalities)}
print('Reasoning weights:', reason_map)

test:   0%|          | 0/7 [00:00<?, ?it/s]

test: 100%|██████████| 7/7 [00:00<00:00, 80.17it/s]

Test Acc 0.4350 Prec 0.4350 Rec 1.0000 F1 0.6063 AUC 0.4914
Reasoning weights: {'static': 3.0762305897047426e-11, 'dynamic': 3.5129698261471276e-08, 'graphical': 6.767016011129459e-14, 'network': 1.0}


**Next steps & notes**
- Replace synthetic data with real feature extraction pipelines for each modality.
- Tune encoder architectures per modality (e.g., CNN for raw images, Transformer/text encoder for dynamic traces).
- Add data augmentation, class weighting, and more robust evaluation (cross-validation).
- Consider modality missingness handling (mask tokens) and interpretability (attention visualization).